# Tutorial 01: Your First Agent - Basic Travel Assistant

##  Learning Objectives
By the end of this notebook, you will:
- Set up your development environment with Python packages
- Understand what an AI agent is
- Create your first agent using Microsoft Agent Framework
- Learn about chat clients and how to configure them with Azure OpenAI
- Send queries to your agent and get responses
- Understand agent instructions and how they guide behavior

##  Prerequisites

Before starting this tutorial, ensure you have:

1. **Python 3.11 or higher** installed
2. **Azure subscription** with Azure OpenAI service enabled
3. **Azure OpenAI deployment** created (gpt-4, gpt-4o, or gpt-35-turbo)
4. **Virtual environment** set up (using venv or uv)
5. **Dependencies installed** from requirements.txt

---

##  Setup Instructions

### Option 1: Using venv (Standard Python)

```bash
# Create virtual environment
python -m venv venv

# Activate it
source venv/bin/activate  # macOS/Linux
# OR
venv\Scripts\activate  # Windows

# Install dependencies
pip install -r requirements.txt
```

### Option 2: Using uv (Faster Alternative)

```bash
# Install uv (if not already installed)
curl -LsSf https://astral.sh/uv/install.sh | sh

# Create virtual environment and install dependencies
uv venv
source .venv/bin/activate  # macOS/Linux
# OR
.venv\Scripts\activate  # Windows

uv pip install -r requirements.txt
```

### Environment Configuration

Create a `.env` file in the project root with your Azure OpenAI credentials:

```bash
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_API_KEY=your-api-key-here
MODEL_DEPLOYMENT_NAME=gpt-4  # or gpt-4o, gpt-35-turbo
```

**Where to find these values:**
- Go to [Azure Portal](https://portal.azure.com)
- Navigate to your Azure OpenAI resource
- Click "Keys and Endpoint" to get your endpoint and API key
- Go to Azure OpenAI Studio → Deployments to find your deployment name

---

## 💡 Key Concepts

### What is an AI Agent?
An AI agent uses an LLM (Large Language Model) to:
1. **Process user inputs** - Understand what users are asking
2. **Make decisions** - Determine the best way to respond
3. **Generate responses** - Provide helpful, contextual answers

### Components You'll Use
- **`ChatAgent`** - The main agent class that handles conversations
- **`AzureOpenAIChatClient`** - Connects your agent to Azure OpenAI models
- **Instructions** - System prompts that define your agent's personality and expertise

---

## Step 1: Verify Installation

Let's verify that all required packages are installed correctly.

In [6]:
# Verify required packages are installed
import sys

required_packages = [
    'agent_framework',
    'dotenv',
    'openai'
]

print("🔍 Checking installed packages...\n")

missing_packages = []
for package in required_packages:
    try:
        __import__(package.replace('-', '_'))
        print(f" {package} - installed")
    except ImportError:
        print(f" {package} - missing")
        missing_packages.append(package)

if missing_packages:
    print(f"\n  Missing packages: {', '.join(missing_packages)}")
    print("Please install them using:")
    print(f"  pip install -r requirements.txt")
else:
    print("\n All required packages are installed!")
    print(f" Python version: {sys.version.split()[0]}")

🔍 Checking installed packages...

 agent_framework - installed
 dotenv - installed
 openai - installed

 All required packages are installed!
 Python version: 3.12.12


## Step 2: Import Required Libraries

Now let's import the Agent Framework components we'll need.

In [12]:
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential

print("✅ Imports successful!")

✅ Imports successful!


## Step 3: Configure Azure OpenAI Credentials

The agent needs access to Azure OpenAI. We'll use **Azure CLI authentication** (managed identity) which is more secure and reliable than API keys.

**📋 Required Environment Variables** (in your `.env` file):
- `AZURE_OPENAI_ENDPOINT`: Your Azure OpenAI endpoint URL
- `MODEL_DEPLOYMENT_NAME`: The name of your deployed model (e.g., gpt-4.1, gpt-4o)

**Authentication:** We use `AzureCliCredential` - make sure you're logged in with `az login`.

In [13]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get Azure OpenAI credentials
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")

# Verify credentials are loaded
if not all([AZURE_OPENAI_ENDPOINT, MODEL_DEPLOYMENT_NAME]):
    print("❌ ERROR: Missing Azure OpenAI configuration!")
    print("\nPlease ensure your .env file contains:")
    print("  AZURE_OPENAI_ENDPOINT=https://your-resource.cognitiveservices.azure.com")
    print("  MODEL_DEPLOYMENT_NAME=gpt-4.1")
else:
    print("✅ Azure OpenAI configuration loaded!")
    print(f"   Endpoint: {AZURE_OPENAI_ENDPOINT}")
    print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

✅ Azure OpenAI configuration loaded!
   Endpoint: https://gk-agent-framework-project.cognitiveservices.azure.com
   Model: gpt-4.1


## Step 4: Create Your First Agent

Let's create a simple Travel Assistant agent. Notice three key components:

1. **Chat Client** - Connects to Azure OpenAI using `AzureOpenAIChatClient` with managed identity
2. **Instructions** - Tells the agent what it is and what it should do
3. **Name** - Optional, but helpful for debugging and multi-agent scenarios

The `AzureOpenAIChatClient` handles all communication with Azure OpenAI, including:
- Managing authentication via Azure CLI credential (no API keys needed!)
- Sending requests to your deployed model
- Handling responses and streaming
- Managing token limits and retries

In [15]:
# Create a chat client that connects to Azure OpenAI using managed identity
credential = AzureCliCredential()

chat_client = AzureOpenAIChatClient(
    endpoint=AZURE_OPENAI_ENDPOINT,
    credential=credential,  # Use Azure CLI credential instead of API key
    deployment_name=MODEL_DEPLOYMENT_NAME
)

# Create the agent
travel_agent = ChatAgent(
    chat_client=chat_client,
    name="TravelAssistant",
    instructions="""
    You are a friendly and knowledgeable travel assistant. Your role is to help users
    plan trips, answer questions about destinations, and provide travel advice.
    
    Guidelines:
    - Be enthusiastic and encouraging about travel
    - Provide specific, actionable recommendations
    - Consider factors like budget, time of year, and traveler interests
    - If you don't know something, be honest about it
    """
)

print("✅ Travel Assistant agent created successfully!")
print(f"   Agent Name: {travel_agent.name}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

✅ Travel Assistant agent created successfully!
   Agent Name: TravelAssistant
   Model: gpt-4.1


## Step 5: Have Your First Conversation

Now let's interact with our agent! We'll use the `run()` method to send queries.

**Note:** The `run()` method is asynchronous, so we use `await`. Jupyter notebooks handle async execution automatically.

In [16]:
# Simple query
query = "What are the best destinations for a beach vacation in Europe?"

print(f"👤 User: {query}\n")

# Run the agent (await because it's async)
response = await travel_agent.run(query)

print(f"🤖 Agent: {response}")

👤 User: What are the best destinations for a beach vacation in Europe?



ServiceResponseException: <class 'agent_framework.azure._chat_client.AzureOpenAIChatClient'> service failed to complete the prompt: Error code: 401 - {'error': {'code': '401', 'message': 'Access denied due to invalid subscription key or wrong API endpoint. Make sure to provide a valid key for an active subscription and use a correct regional API endpoint for your resource.'}}

## Step 6: Try More Queries

Let's ask different types of questions to see how our agent responds based on the instructions we provided.

In [ ]:
# Ask about a specific destination
query = "Tell me about visiting Tokyo. What should I see?"

print(f"👤 User: {query}\n")
response = await travel_agent.run(query)
print(f"🤖 Agent: {response}")

User: Tell me about visiting Tokyo. What should I see?

Agent: Tokyo is a fantastic city—truly one of the world’s most exciting destinations! Whether you love art, food, shopping, or exploring unique neighborhoods, there’s something for everyone. Here are some must-see spots and tips for a great trip:

### Must-See Neighborhoods & Sights

**1. Shibuya**
- Famous for the iconic Shibuya Crossing—the world’s busiest pedestrian crossing.
- Vibrant shopping and dining, the Hachiko statue, and nightlife.

**2. Shinjuku**
- Explore Shinjuku Gyoen National Garden (gorgeous, especially during cherry blossom season).
- Check out the views from the Tokyo Metropolitan Government Building (free observation decks).
- Enjoy Omoide Yokocho and Golden Gai for atmospheric bars and local eats.

**3. Asakusa**
- Home to Senso-ji, Tokyo’s oldest temple. The surrounding streets are full of traditional shops and street foods.
- Take a stroll along the Sumida River, or hop on a river cruise!

**4. Akihabara**

In [ ]:
# Ask for specific recommendations
query = "I have 5 days and a $2000 budget. Where should I go from New York?"

print(f"👤 User: {query}\n")
response = await travel_agent.run(query)
print(f"🤖 Agent: {response}")

User: I have 5 days and a $2000 budget. Where should I go from New York?

Agent: That’s awesome! With 5 days and a $2,000 budget, you have some fantastic options for a memorable trip from New York. Here are several ideas based on different interests:

---

### 1. Montreal, Canada  
**Why Go**: It's close, offers a mix of European charm and North American energy, and is great for food, culture, and festivals.

- **Travel**: Take an Amtrak train or fly (flights are usually under 2 hours; round-trip can often be found for $250–$400).
- **Accommodation**: Boutique hotels or Airbnbs ($150–$200/night).
- **Must-Do**: Explore Old Montreal, visit the Jean-Talon Market, try local cafes and poutine, climb Mont Royal.
- **Tip**: Bring your passport!

---

### 2. Charleston, South Carolina  
**Why Go**: Gorgeous historic streets, excellent food, and southern charm.

- **Travel**: Direct flights (~2.5 hours; round-trip $200–$350).
- **Accommodation**: Historic inns or downtown hotels ($180–$220/nig

In [ ]:
# Ask for travel tips
query = "What should I pack for a winter trip to Iceland?"

print(f"👤 User: {query}\n")
response = await travel_agent.run(query)
print(f"🤖 Agent: {response}")

User: What should I pack for a winter trip to Iceland?

Agent: How exciting—a winter trip to Iceland! Iceland’s winter weather can be unpredictable, with snow, wind, and chilly temperatures, but a little planning can make your adventure comfortable and magical. Here’s a detailed packing list to help you:

### **Clothing**
- **Thermal base layers:** Pack moisture-wicking long-sleeve shirts and leggings; Merino wool is great!
- **Warm sweaters:** Wool or fleece works well as mid-layers.
- **Waterproof and windproof outerwear:** A good insulated jacket and waterproof pants are essential for staying dry and warm.
- **Sturdy waterproof boots:** Snow and ice are common; boots with good grip are a must.
- **Thick socks:** Wool socks keep feet warm and dry.
- **Gloves/mittens:** Consider waterproof gloves for extra warmth.
- **Warm hat:** Covers your ears!
- **Scarf or neck gaiter:** Essential to protect against icy winds.
- **Swimsuit:** For hot springs like the Blue Lagoon or local geotherma

## 📊 Understanding What Just Happened

Let's break down the flow:

```
1. User Query → 2. ChatAgent → 3. AzureOpenAIChatClient → 4. Azure OpenAI API
                                                               ↓
8. Display ← 7. ChatAgent ← 6. AzureOpenAIChatClient ← 5. LLM Response
```

**Key Points:**
- Each `run()` call is **independent** - the agent doesn't remember previous conversations (yet!)
- The **instructions** guide how the agent responds to ALL queries
- The **AzureOpenAIChatClient** handles all API communication automatically
- The agent uses the LLM's knowledge up to its training cutoff date

**What the Framework Does for You:**
- ✅ Manages API authentication and requests
- ✅ Handles token counting and limits
- ✅ Formats messages in the correct structure
- ✅ Retries on failures
- ✅ Streams responses efficiently

## 🔬 Experiment: Change the Instructions

Let's see how instructions affect behavior. We'll create a more casual version of the agent using the same Azure OpenAI connection.

In [ ]:
# Create a more casual travel buddy using the same chat_client
casual_agent = ChatAgent(
    chat_client=chat_client,  # Reusing the same Azure OpenAI connection
    name="CasualTravelBuddy",
    instructions="""
    You're a chill travel buddy who's been everywhere and loves sharing stories.
    Keep it casual, use emojis, and give honest opinions. Don't be too formal!
    """
)

query = "What are the best destinations for a beach vacation in Europe?"

print(f"👤 User: {query}\n")
response = await casual_agent.run(query)
print(f"😎 Casual Agent: {response}")

User: What are the best destinations for a beach vacation in Europe?

Casual Agent: Oh man, Europe is packed with epic beach spots! 😎🌊 Depends what vibe you're after, but here are a few faves:

**1. Algarve, Portugal**  
Golden cliffs, secret coves, and just laid-back vibes. Surfers, foodies, and sun-lovers all fit in here. Lagos is super fun, but there are quieter spots too!

**2. Greek Islands (Santorini, Mykonos, Crete...)**  
Crystal clear water, blue domes, killer sunsets. Santorini is romantic, Mykonos has parties, Crete’s got wild beaches (Elafonissi is dreamy).

**3. Amalfi Coast, Italy**  
Okay, technically beaches are small (sometimes just pebbles), but the views? Unreal. Positano is pure Instagram goals. Try Sorrento for fewer crowds.

**4. Costa Brava, Spain**  
You get cliffside villages, turquoise bays, and some killer seafood. Tossa de Mar is a must for chill beach vibes.

**5. French Riviera (Nice, Antibes, St. Tropez...)**  
Glitz & glamour mixed with med beaches. More

**💡 Notice:** The **same query** gets a different tone and style based on instructions! The instructions act as the agent's "personality" and guide how it interprets and responds to user requests.

## ✅ Key Takeaways

1. **Setup matters**: Proper environment configuration with venv/uv and `.env` files is essential
2. **Azure OpenAI integration**: The `AzureOpenAIChatClient` handles all API communication
3. **Creating agents is simple**: Just provide a chat client and instructions
4. **Instructions are powerful**: They define personality, expertise, and behavior
5. **Agents are stateless by default**: Each `run()` is independent (we'll fix this in Tutorial 03)
6. **The framework abstracts complexity**: You don't deal with raw API calls, token management, etc.

## ⚠️ Current Limitations

Our agent is basic and has limitations:
- ❌ Can't call external APIs (weather, flights, etc.)
- ❌ Doesn't remember previous messages in a conversation
- ❌ Can't verify real-time information
- ❌ Limited to the LLM's training data

**In the next tutorial**, we'll add **Tools** to let the agent call functions and get real-time data!

---

## 🚀 What's Next?

In **Tutorial 02: Agent with Tools**, you'll learn to:
- ✅ Add function calling capabilities
- ✅ Create tools for weather checking and currency conversion
- ✅ Let the agent decide when to use tools
- ✅ Handle tool errors and responses

**Ready?** → Open `02_agent_with_tools.ipynb`

---

## 📚 Additional Resources

- [Azure OpenAI Service Documentation](https://learn.microsoft.com/azure/ai-services/openai/)
- [Agent Framework Documentation](https://learn.microsoft.com/agent-framework/)
- [Python Environment Management](https://docs.python.org/3/library/venv.html)
- [UV Package Manager](https://github.com/astral-sh/uv)